github repo: 
google drive: https://drive.google.com/drive/folders/1z4VBvPry4pktrcovTjSJVmrh1wcfZqkB?usp=sharing


# Problem (data transformation design): Q2a. Design a data transformation (10 points)

I change the reviews so they look more like something someone would write quickly in a text or an online comment. I use slang, abbreviations, word mix-ups, spelling mistakes, and a few adjective synonyms. A person should usually still understand whether the review is positive or negative, even though the wording looks different to the model because its probably rarely used in online texts like research papers/blogs/posts etc.


## 1. Phrase/spelling replacements

I replace some standard words and phrases with slang or abbreviations we use when speaking or texting. A person would usually understand these because we use them in daily conversations. They are more common in casual messages than in formal writing, and the model may recognize them less reliably than the standard spelling. 

**Examples:**

- `probably` → `prolly`
- `because` → `cuz`
- `very` → `hella`
- `you` → `u`, `your` → `ur`, and `before` → `b4`
- `for real` → `frfr` and `for sure` → `fsfs`
- `is not` → `ain't`

I also include a few common word-choice mistakes that we as humans will make, such as `better than` → `better then` and `the effect` → `the affect`. These are different from slang, but a reader can often figure out what was intended from the sentence.

Also, each matching phrase has a **50% chance** of being replaced. Matching ignores capitalization and checks longer phrases first, so a phrase such as `the effect` is handled together. I preserve the original capitalization where possible.


## 2. Word mix-ups and common misspellings

I replace some review words with a similar-looking or similar-sounding word. Some replacements are real words, but they are used in the wrong context. For example in a movie review, a person can usually tell that `grate` was meant to be `great`.

**Examples:** `great` → `grate`, `poor` → `pore`, `scene` → `seen`, `cast` → `cat`, and `acting` → `acing`.

An unprotected word in this list has a **90% chance** of receiving its fixed replacement. The surrounding review is still there to help a reader understand it, but the model now sees a different spelling or even a different word.


## 3. Recognizable spellings of sentiment words

I also change the spelling of words that express how someone feels about the movie. These are intended to look like informal spellings or ordinary spelling mistakes, while still being recognizable.

**Examples:** `good` → `gud`, `loved` → `luved`, `wonderful` → `wunderful`, `terrible` → `terible`, and `boring` → `borring`.

An unprotected word in this list has an **80% chance** of receiving its fixed spelling. For example, “I luved it” still sounds positive, and “It was terible” still sounds negative. I am changing how the opinion is written rather than intentionally changing the opinion itself.


## 4. Random typing errors

I also add typing errors to other words to simulate human typing errors. 

I randomly choose one of these edits:

- Swap two neighboring letters inside a word, such as `watch` → `wtach`.
- Drop an internal letter, such as `watch` → `wach`.
- Repeat an internal letter, such as `place` → `placce`.
- Replace a vowel with a nearby consonant key, such as `story` → `stlry`.
- For words in the review/sentiment list, replace a letter with a number: `o` → `0`, `i` → `1`, `e` → `3`, or `a` → `4`. For example, `bad` could become `b4d`.

These are examples of possible edits, rather than a fixed replacement for each word. Other words in the code's review/sentiment list have an **80% chance** of a random edit. Other unprotected words with at least four letters have a **40% chance**. Each selected word gets at most one edit in this step. Three-letter words in the review/sentiment list only use vowel or number substitutions. Longer words in that list can use all five edit types. For other words, choosing the number-substitution mode falls back to a nearby-key vowel typo. If an edit cannot be applied, the code tries a vowel substitution instead, or leaves the word unchanged if none is possible.

The nearby-key choices are:

| Vowel | Replacement keys |
|---|---|
| `a` | `q`, `w`, `s`, `z` |
| `e` | `w`, `s`, `d`, `r` |
| `i` | `j`, `k`, `l` |
| `o` | `k`, `l`, `p` |
| `u` | `y`, `h`, `j` |

I protect words such as `not` and `never` from these typos because changing them could change the sentiment. Words introduced by the phrase/slang replacements are protected too, so an abbreviation such as `prolly` is not immediately changed again.


## 5. Adjective synonym replacement

I use NLTK to identify adjectives and WordNet to find alternative words. This tests whether the model can understand a similar description written with different vocabulary.

**Examples:** `small town` → `little town` and `quick train` → `speedy train`.

I randomly choose **up to two eligible adjective occurrences per review**. I only use adjectives tagged `JJ`, and skip words in the protected or review/sentiment lists. This avoids directly replacing many of the obvious positive or negative words.

For each candidate, I use the first WordNet adjective meaning and choose a different, single-word synonym containing only letters. The synonym cannot be more than four letters longer than the original word. If there is no suitable synonym, I skip that adjective. I preserve the original capitalization when inserting the new word.


## How the steps work together

The code applies **WordNet synonyms first, phrase replacements second, and word-level typos last**. In the final step, protected words are skipped first, then the fixed word-mix-up list is checked, then the sentiment-spelling list, and then the random typing rules. If a fixed replacement is not selected, that word can still receive a later random typo.

I use a separate random generator for each review, seeded with the SHA-256 digest of `"q2-combined-v5:" + original_text`. This means the same review always gets the same transformed text, even if the dataset order changes. The same function can therefore be reused for Q3 training augmentation.

The synonym and typo steps skip HTML tags and do not match parts of contractions. Phrase matching still runs on the raw text. There is no minimum number of edits, so a short review might change very little, while a longer review might contain several changes.


## Examples from the final implementation

**Original:** I really loved this movie because the acting was very good.

**Transformed:** I rly luved this movie cuz the acing was hella gud.

The opinion is still positive. The text combines abbreviations, informal phrasing, recognizable misspellings, and a word mix-up.

**Original:** The plot was boring and the acting was terrible.

**Transformed:** The plod was borring and the acing was terible.

The opinion is still negative, even though several words are misspelled.

## Why this is reasonable

People do not always write movie reviews in standard English. They use slang, shorten words, mix up similar words, or make typing mistakes, especially when writing quickly. These changes are things we could encounter in a real review, and a reader can often recover the intended meaning from the rest of the sentence.

There are still limitations. A WordNet synonym may not fit the exact meaning in the sentence, and a word mix-up such as `cast` → `cat` can be ambiguous. Several edits together can also make a review harder to read. Keeping the label unchanged is the intention of the transformation, not a guarantee that every transformed example keeps exactly the same meaning. I use the positive and negative wording around the edits to judge whether the review's sentiment is still clear.


## Q2b: Accuracy before and after the transformation

I first evaluated the same fine-tuned BERT model on the original test reviews and on the transformed versions. This checks how changing the writing style affects the model before adding transformed examples to training.

The tables below are calculated from the saved prediction files. Each file stores a prediction followed by its true label, and each evaluation contains the same 25,000 test examples.


In [1]:
from pathlib import Path
from IPython.display import Markdown, display

part1_dir = next(path.resolve() for path in
                 (Path("."), Path("part1"), Path("nyu-dsga1011-a4/part1"))
                 if (path / "out_original.txt").is_file())

def read_predictions(filename):
    values = [int(line) for line in (part1_dir / filename).read_text().splitlines()]
    assert len(values) == 50000
    return values[::2], values[1::2]

filenames = {
    "baseline_original": "out_original.txt",
    "baseline_transformed": "out_transformed.txt",
    "augmented_original": "out_augmented_original.txt",
    "augmented_transformed": "out_augmented_transformed.txt",
}
predictions = {}
accuracies = {}
labels = None
for name, filename in filenames.items():
    pred, reference = read_predictions(filename)
    if labels is None:
        labels = reference
    assert labels == reference
    predictions[name] = pred
    accuracies[name] = sum(a == b for a, b in zip(pred, reference)) / len(reference)

lines = ["### Original vs. transformed input: same model",
         "", "| Test input | Correct predictions | Accuracy |",
         "|---|---:|---:|"]
for name, label in [("baseline_original", "Original reviews"),
                    ("baseline_transformed", "Transformed reviews")]:
    correct = sum(a == b for a, b in zip(predictions[name], labels))
    lines.append(f"| {label} | {correct:,} / {len(labels):,} | {accuracies[name] * 100:.3f}% |")
display(Markdown("\n".join(lines)))

lines = ["### Before vs. after training with data augmentation", "",
         "| Training data | Original test accuracy | Transformed test accuracy |",
         "|---|---:|---:|",
         f"| Original training reviews | {accuracies['baseline_original']*100:.3f}% | {accuracies['baseline_transformed']*100:.3f}% |",
         f"| Original reviews + 5,000 transformed training reviews | {accuracies['augmented_original']*100:.3f}% | {accuracies['augmented_transformed']*100:.3f}% |",
         f"| Change (percentage points) | {(accuracies['augmented_original']-accuracies['baseline_original'])*100:+.3f} | {(accuracies['augmented_transformed']-accuracies['baseline_transformed'])*100:+.3f} |"]
display(Markdown("\n".join(lines)))


### Original vs. transformed input: same model

| Test input | Correct predictions | Accuracy |
|---|---:|---:|
| Original reviews | 23,202 / 25,000 | 92.808% |
| Transformed reviews | 20,329 / 25,000 | 81.316% |

### Before vs. after training with data augmentation

| Training data | Original test accuracy | Transformed test accuracy |
|---|---:|---:|
| Original training reviews | 92.808% | 81.316% |
| Original reviews + 5,000 transformed training reviews | 92.736% | 88.724% |
| Change (percentage points) | -0.072 | +7.408 |

### Why I changed the first transformation

My first version did not really affect the model that much. The transformed accuracy was still around **90%**, compared with **92.808%** on the original reviews, so the decrease was less than the required four percentage points. The earlier version used phrase replacements, one adjective synonym, and a lower rate of vowel typos. Some reviews may not have contained the phrases in the dictionary, and the model could still handle many of the small edits.

I then added more kinds of word-level mistakes, including word mix-ups, recognizable misspellings of sentiment words, and different typing errors. I also increased the typo probabilities and allowed up to two eligible adjective replacements. The final transformation reduced accuracy from **92.808% to 81.316%**.

The earlier “around 90%” figure is an approximate observation from that trial. The exact final numbers in the tables come from the complete saved prediction files.


## Q3: Data augmentation — report and analysis

For augmentation, I kept all 25,000 original training reviews and added transformed versions of 5,000 randomly selected training reviews. This gave 30,000 training examples. The extra examples came from the training set, and the original and transformed versions kept the same sentiment label.

### 1. Did augmentation improve accuracy on transformed reviews?

Yes. Accuracy on the transformed test set increased from **81.316% to 88.724%**, an improvement of **7.408 percentage points**, or **1,852 more correct predictions** overall.

I think this happens because the model gets practice with the same kinds of changes during training. It sees slang, misspellings, and typing errors together with the correct sentiment label, so it has a better chance of recognizing the intended opinion when similar changes appear at evaluation time. It still does not recover all the original performance: 88.724% is lower than the 92.808% accuracy on the unchanged reviews.

### 2. What happened to accuracy on the original reviews?

It went from **92.808% to 92.736%**... **18 fewer correct predictions**. Augmentation did not improve original-test accuracy in this run, but the difference is very small. I would not treat it as a major loss in performance.

Keeping the original training reviews probably helped the model retain its performance on normal inputs, while the extra examples helped it handle the transformed ones. This is a possible explanation for the pattern we observed; this single comparison does not separate the effects of the extra examples, additional training updates, and normal training variation.


### Examples that became errors after transformation

Below shows that our transformed text was generated using the same final `custom_transform()` used for Q2b and Q3, and the predictions below come from the saved evaluation files. Indices start at 0; label `0` means negative and `1` means positive. In both cases, the original model was correct on the original text and wrong on the transformed text.


In [2]:
examples = [{'test_index': 14758,
  'label': 1,
  'original': 'This is a great movie. Too bad it is not available on home video.',
  'transformed': 'This is a grate movie. Too bad it is not availlable on hme vidro.',
  'pred_original': 1,
  'pred_transformed': 0,
  'pred_augmented_transformed': 1},
 {'test_index': 6540,
  'label': 0,
  'original': "I can't believe this movie has an average rating of 7.0! It is a fiendishly bad "
              'movie, and I saw it when it was fairly new, and I was in the age group that is '
              'supposed to like it!',
  'transformed': "I can't believe this movie has an maen ratnig of 7.0! It is a fiendishly bqd "
                 'movie, and I saw it wen it was fairly new, and I was in the age grkup that is '
                 'supposd to lkie it!',
  'pred_original': 0,
  'pred_transformed': 1,
  'pred_augmented_transformed': 1}]

label_names = {0: "negative", 1: "positive"}
for example in examples:
    i = example["test_index"]
    assert labels[i] == example["label"]
    assert predictions["baseline_original"][i] == example["pred_original"]
    assert predictions["baseline_transformed"][i] == example["pred_transformed"]
    assert predictions["augmented_transformed"][i] == example["pred_augmented_transformed"]
    display(Markdown(f"#### Test example {i}: true label = {label_names[example['label']]}"))
    display(Markdown("**Original:** " + example["original"]))
    display(Markdown("**Transformed:** " + example["transformed"]))
    lines = ["| Evaluation | Prediction | Correct? |", "|---|---|---|"]
    for title, key in [("Original model, original text", "pred_original"),
                       ("Original model, transformed text", "pred_transformed"),
                       ("Augmented model, transformed text", "pred_augmented_transformed")]:
        prediction = example[key]
        lines.append(f"| {title} | {label_names[prediction]} | {'Yes' if prediction == example['label'] else 'No'} |")
    display(Markdown("\n".join(lines)))


#### Test example 14758: true label = positive

**Original:** This is a great movie. Too bad it is not available on home video.

**Transformed:** This is a grate movie. Too bad it is not availlable on hme vidro.

| Evaluation | Prediction | Correct? |
|---|---|---|
| Original model, original text | positive | Yes |
| Original model, transformed text | negative | No |
| Augmented model, transformed text | positive | Yes |

#### Test example 6540: true label = negative

**Original:** I can't believe this movie has an average rating of 7.0! It is a fiendishly bad movie, and I saw it when it was fairly new, and I was in the age group that is supposed to like it!

**Transformed:** I can't believe this movie has an maen ratnig of 7.0! It is a fiendishly bqd movie, and I saw it wen it was fairly new, and I was in the age grkup that is supposd to lkie it!

| Evaluation | Prediction | Correct? |
|---|---|---|
| Original model, original text | negative | Yes |
| Original model, transformed text | positive | No |
| Augmented model, transformed text | positive | No |

In **example 14758**, the review is positive about the movie; the complaint is that it is unavailable on home video. After `great` becomes `grate`, the clearest positive word is less obvious, while “Too bad” and “not available” are still present. This may explain why the original model switches from positive to negative. After augmentation, the model predicts positive correctly again.

In **example 6540**, the reviewer is saying the movie is bad even though it has a 7.0 average rating. The transformation changes `bad` to `bqd` and introduces other errors, including `average rating` becoming `maen ratnig`. The original model switches from negative to positive, and the augmented model still gets this example wrong. 

### 3. One limitation of this augmentation approach

The model mostly gets practice with the transformation I designed. That does not mean it will handle every unseen condition. It may still struggle with new slang, sarcasm, different writing styles, or spelling errors that are unlike the ones in my rules.

I also only add one transformed version of each of the 5,000 selected training reviews, so the model does not see every possible combination of changes. The improvement shows that it adapts better to this particular transformed test set, rather than proving it is robust to all out-of-distribution inputs. Example 6540 is one case where the error remains even after augmentation.
